# Phase 2 — Results and Benchmarking: Multimodal Land Cover Segmentation

## Research Question

> *Does conditioning a segmentation transformer on AI-generated scene descriptions improve land cover segmentation accuracy compared to an image-only baseline?*

## What This Notebook Does

This notebook extends Phase 1 to the full 10,000-sample dataset. Dataset exploration was already
completed in Phase 1 and is not repeated here. The focus is on:

1. **Full training pipeline** on the complete dataset (80/10/10 split, seeded)
2. **Three experiments**: image-only baseline, image + Gemma captions, image + Qwen captions
3. **Class-weighted loss** to address the dominant-class prediction problem identified in Phase 1
4. **Experiments with configurable backbone** — switch between MiT-B0 through MiT-B5 via config, this notebook will include couple of experiments with other backbones

Larger backbones may outperform B0 —> this is an ablation opportunity.

Colab was using to run this notebook, L4 GPU was used.
All results were logging in W&B

In [ ]:
!pip install -q torch torchvision transformers accelerate evaluate
!pip install -q pillow matplotlib seaborn scikit-learn pandas numpy pyyaml
!pip install -q wandb


In [ ]:
!nvidia-smi

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import wandb
wandb.login(key='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')

In [ ]:
import os
import platform
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from PIL import Image
from sklearn.model_selection import train_test_split

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

from transformers import (
    SegformerForSemanticSegmentation,
    SegformerImageProcessor,
    BertModel,
    BertTokenizer,
)
import wandb

warnings.filterwarnings('ignore')
print('All imports successful.')
print(f'PyTorch version: {torch.__version__}')


In [ ]:
# CONFIG
# Re-run with a different CONFIG
# Each experiment logged to W&B under a new run name
CONFIG = {
    # Experiment
    'run_name':          'baseline-mit-b0',   # descriptive name logged to W&B
    'caption_col':       None,                # None=image-only - 'vision_gemma3-4b' - 'vision_qwen3-vl-8b'

    # Data
    'image_size':        512,
    'batch_size':        4,
    'seed':              42,

    # Model
    # Backbone options: nvidia/mit-b0 (3.7M), mit-b1 (14M), mit-b2 (27M) and so on.
    'backbone':          'nvidia/mit-b0',
    'text_model':        'bert-base-uncased',
    'freeze_bert':       True,    # unfreeze as Phase 3 ablation
    'num_classes':       7,

    # Training
    'num_epochs':        20,
    'learning_rate':     6e-5,
    'weight_decay':      0.01,
    'grad_accumulation': 2,      # effective batch size = batch_size x grad_accumulation = 8
    # Important NOTE: class-weighted loss addresses the dominant-class prediction
    # problem identified in Phase 1 — the model was covering the whole image with
    # the most frequent class to minimise plain cross-entropy.
    # We need to careful about it by using class weights
    'use_class_weights': True,
    # fp16 mixed precision: only enabled on CUDA
    'mixed_precision':   True,

    # W&B
    'use_wandb':         True,
    'wandb_project':     'multimodal-landcover-segmentation',
    'wandb_entity':      'ftmoztl',   # set to your W&B username, or leave None

    # Save checkpoints
    'save_dir':          (
        '/content/drive/MyDrive/multimodal-landcover-segmentation/checkpoints'
        if os.path.exists('/content/drive/MyDrive')
        else 'checkpoints'
    ),
    'save_best':         True,   # save checkpoint when val mIoU improves
}

# Backbone → last-stage channel dimension (used to size the fusion module)
BACKBONE_CHANNELS = {
    'nvidia/mit-b0': 256,
    'nvidia/mit-b1': 512,
    'nvidia/mit-b2': 512,
    'nvidia/mit-b3': 512,
    'nvidia/mit-b4': 512,
    'nvidia/mit-b5': 512,
}

# Caption columns — the only ones without label leakage
FAIR_CAPTION_COLS = ['vision_gemma3-4b', 'vision_qwen3-vl-8b']


def get_device():
    """
    I'm Macbook user BUT: MPS (Apple Silicon) crashes during SegFormer backprop due to a
    known non-contiguous tensor bug in the autograd engine. CPU is used on macOS.
    Colab use CUDA and are not affected.
    """
    if torch.cuda.is_available():
        return torch.device('cuda')
    elif platform.system() == 'Darwin':
        return torch.device('cpu')  # MPS incompatible with SegFormer backward
    return torch.device('cpu')


device = get_device()
# num_workers=0 for MAcbook, 2 for Colab
NUM_WORKERS = 0 if platform.system() == 'Darwin' else 2

torch.manual_seed(CONFIG['seed'])
np.random.seed(CONFIG['seed'])

print(f'Device      : {device}')
print(f'num_workers : {NUM_WORKERS}')
print(f'Backbone    : {CONFIG["backbone"]}')
print(f'caption_col : {CONFIG["caption_col"]}')
print(f'save_dir    : {CONFIG["save_dir"]}')


In [ ]:
CLASS_INFO = {
    0: {'name': 'Tree',     'color': (0,   100, 0)},
    1: {'name': 'Shrub',    'color': (255, 182, 193)},
    2: {'name': 'Grass',    'color': (154, 205, 50)},
    3: {'name': 'Crop',     'color': (255, 215, 0)},
    4: {'name': 'Built-up', 'color': (139, 69,  19)},
    5: {'name': 'Barren',   'color': (211, 211, 211)},
    6: {'name': 'Water',    'color': (0,   0,   255)},
}
CLASS_COLORS = {info['color']: idx for idx, info in CLASS_INFO.items()}
CLASS_NAMES  = [CLASS_INFO[i]['name'] for i in range(len(CLASS_INFO))]
id2label     = {idx: info['name'] for idx, info in CLASS_INFO.items()}
label2id     = {info['name']: idx for idx, info in CLASS_INFO.items()}
print('Classes:', CLASS_NAMES)


## 1. Data Pipeline

Dataset exploration was completed in Phase 1 and is not repeated here. I load the full 10,000-sample
dataset, apply a seeded 80/10/10 split in memory, and compute inverse-frequency class weights to
address the dominant-class prediction problem noted by the Phase 1 reviewer.


In [ ]:
for _c in [
    '/content/drive/MyDrive/multimodal-landcover-segmentation/data',
    'data',
    '../data',
]:
    if os.path.exists(os.path.join(_c, 'captions.csv')):
        DATA_DIR = _c
        break
else:
    DATA_DIR = 'data'

IMAGES_DIR   = os.path.join(DATA_DIR, 'images')
MASKS_DIR    = os.path.join(DATA_DIR, 'masks')
CAPTIONS_CSV = os.path.join(DATA_DIR, 'captions.csv')
print(f'Data directory: {os.path.abspath(DATA_DIR)}')


def normalise_filename(fn_raw):
    """Ensure filename ends with .png — works for any digit count."""
    fn = str(fn_raw).strip()
    return fn if fn.endswith('.png') else fn + '.png'


def rgb_mask_to_class(mask_rgb: np.ndarray) -> np.ndarray:
    """Convert RGB mask (H, W, 3) to class index mask (H, W) using vectorised NumPy."""
    class_mask = np.zeros(mask_rgb.shape[:2], dtype=np.int64)
    for color, class_idx in CLASS_COLORS.items():
        class_mask[np.all(mask_rgb == np.array(color, dtype=np.uint8), axis=-1)] = class_idx
    return class_mask


def class_mask_to_rgb(class_mask: np.ndarray) -> np.ndarray:
    """Convert class index mask back to RGB for visualization."""
    rgb = np.zeros((*class_mask.shape, 3), dtype=np.uint8)
    for idx in range(len(CLASS_INFO)):
        rgb[class_mask == idx] = CLASS_INFO[idx]['color']
    return rgb


In [ ]:
class LandCoverDataset(Dataset):
    """
    Designed this dataset class to handle the three-way correspondence between
    satellite images, segmentation masks, and (optionally) text captions.
    Setting caption_col=None gives the image-only baseline DataLoader;
    setting it to a column name gives the fused model DataLoader.
    """

    def __init__(self, df, images_dir, masks_dir, processor,
                 caption_col=None, image_size=512):
        """
        Args:
            df:          DataFrame subset (already filtered by split)
            images_dir:  path to folder containing satellite images (e.g. data/images/)
            masks_dir:   path to folder containing RGB mask images  (e.g. data/masks/)
            processor:   SegformerImageProcessor — handles resizing and normalization
            caption_col: which caption column to load (None = image-only mode)
            image_size:  target image height and width in pixels
        """
        self.df          = df.reset_index(drop=True)
        self.images_dir  = images_dir
        self.masks_dir   = masks_dir
        self.processor   = processor
        self.caption_col = caption_col
        self.image_size  = image_size

        # Identify the filename column
        self.fn_col      = df.columns[0] # fallback
        for c in ['filename', 'image_id', 'image_name', 'file_name', 'id', 'name']:
            if c in df.columns:
                self.fn_col = c
                break

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row      = self.df.iloc[idx]

        # normalise_filename
        filename = normalise_filename(row[self.fn_col])

        # Load and preprocess satellite image
        image = Image.open(os.path.join(self.images_dir, filename)).convert('RGB')
        proc  = self.processor(
            images=image,
            size={'height': self.image_size, 'width': self.image_size},
            return_tensors='pt'
        )
        pixel_values = proc['pixel_values'].squeeze(0)

        # Load, resize, and convert mask to class indices
        mask_rgb = np.array(
            Image.open(os.path.join(self.masks_dir, filename))
            .convert('RGB')
            .resize((self.image_size, self.image_size), Image.NEAREST)
        )
        label = torch.tensor(rgb_mask_to_class(mask_rgb), dtype=torch.long)

        item = {'pixel_values': pixel_values, 'labels': label, 'filename': filename}

        # Optionally include a caption string for the fused model experiments
        if self.caption_col and self.caption_col in self.df.columns:
            item['caption'] = str(row[self.caption_col])
        return item


#### The problem (dominant-class prediction):

In land cover data, some classes appear much more frequently than others. For example, if 60% of all pixels are "Tree" and only 2% are "Water", a lazy model can get decent accuracy by just predicting "Tree" everywhere — it's right 60% of the time without learning anything useful.

Standard cross-entropy loss treats all pixels equally, so the model is rewarded more for getting the common class right and ignores rare classes.

Inverse-frequency weighting —> the fix:
* Give rare classes a higher penalty when the model gets them wrong. The weight is the inverse of how often the class appears

In [ ]:
captions_df = pd.read_csv(CAPTIONS_CSV)
print(f'Dataset: {len(captions_df)} samples   Columns: {list(captions_df.columns)}')

# Seeded 80/10/10 split
all_idx = captions_df.index.tolist()
train_idx, temp_idx = train_test_split(all_idx, test_size=0.20, random_state=CONFIG['seed'])
val_idx,   test_idx = train_test_split(temp_idx,  test_size=0.50, random_state=CONFIG['seed'])

train_df = captions_df.loc[train_idx]
val_df   = captions_df.loc[val_idx]
test_df  = captions_df.loc[test_idx]

print(f'Train: {len(train_df):,}  Val: {len(val_df):,}  Test: {len(test_df):,}')

processor = SegformerImageProcessor.from_pretrained(
    CONFIG['backbone'], do_reduce_labels=False
)


def create_loaders(config, train_df, val_df, test_df, processor):
    """Create DataLoaders for a given config. caption_col is read from config."""
    caption_col = config.get('caption_col', None)
    pin = (device.type == 'cuda')
    kwargs = dict(image_size=config['image_size'])

    train_ds = LandCoverDataset(train_df, IMAGES_DIR, MASKS_DIR, processor,
                                caption_col=caption_col, **kwargs)
    val_ds   = LandCoverDataset(val_df,   IMAGES_DIR, MASKS_DIR, processor,
                                caption_col=caption_col, **kwargs)
    test_ds  = LandCoverDataset(test_df,  IMAGES_DIR, MASKS_DIR, processor,
                                caption_col=caption_col, **kwargs)

    tl = DataLoader(train_ds, batch_size=config['batch_size'], shuffle=True,
                    num_workers=NUM_WORKERS, pin_memory=pin)
    vl = DataLoader(val_ds,   batch_size=config['batch_size'], shuffle=False,
                    num_workers=NUM_WORKERS, pin_memory=pin)
    el = DataLoader(test_ds,  batch_size=config['batch_size'], shuffle=False,
                    num_workers=NUM_WORKERS, pin_memory=pin)
    return tl, vl, el


# Class weights ---> computed once, shared across all experiments)
# I use the composition percentage columns from the CSV to estimate class frequencies.
# This is much faster than iterating through 8,000 mask files and is accurate enough
# because the percentages are derived from the actual masks.
pct_cols = [c for c in CLASS_NAMES if c in captions_df.columns]

if len(pct_cols) == CONFIG['num_classes']:
    mean_pcts = train_df[pct_cols].mean().values / 100.0
    raw_weights = 1.0 / (mean_pcts + 1e-6)
    class_weights = torch.tensor(
        raw_weights / raw_weights.sum() * CONFIG['num_classes'],
        dtype=torch.float32
    )
    print('Class weights from composition columns:')
else:
    # Fallback: uniform weights
    class_weights = torch.ones(CONFIG['num_classes'])
    print('Composition columns not found — using uniform weights.')

for i, (name, w) in enumerate(zip(CLASS_NAMES, class_weights)):
    print(f'  [{i}] {name:10s}  weight = {w:.3f}')


## 2. Model Architecture

All three model variants (image-only baseline, fused with Gemma captions, fused with Qwen captions)
share the same model structure (build_model). Changing backbone and caption column
is all that is needed to switch experiments — the channel dimensions and fusion parameters
are derived automatically from "BACKBONE_CHANNELS".


In [ ]:
class CrossAttentionFusion(nn.Module):
    """
    Injects text features into the visual pipeline via cross-attention.
    Image features are queries; BERT token embeddings are keys and values.
    The output is the same shape as the image features so it drops into
    the SegFormer encoder pipeline without changing the decoder.
    """

    def __init__(self, image_dim, text_dim=768, hidden_dim=256, num_heads=4):
        super().__init__()
        self.q_proj     = nn.Linear(image_dim, hidden_dim)
        self.k_proj     = nn.Linear(text_dim,  hidden_dim)
        self.v_proj     = nn.Linear(text_dim,  hidden_dim)
        self.cross_attn = nn.MultiheadAttention(hidden_dim, num_heads, batch_first=True)
        self.out_proj   = nn.Linear(hidden_dim, image_dim)
        self.norm       = nn.LayerNorm(image_dim)

    def forward(self, image_features, text_features):
        """
        image_features : (B, H*W, image_dim)
        text_features  : (B, seq_len, 768)
        returns        : (B, H*W, image_dim)
        """
        Q = self.q_proj(image_features)
        K = self.k_proj(text_features)
        V = self.v_proj(text_features)
        attn_out, attn_w = self.cross_attn(Q, K, V)
        return self.norm(image_features + self.out_proj(attn_out)), attn_w


class SegformerWrapper(nn.Module):
    """Thin wrapper giving SegformerForSemanticSegmentation the same forward
    interface as MultimodalSegformer: forward(pixel_values, captions=None) -> logits."""

    def __init__(self, model):
        super().__init__()
        self.model = model

    def forward(self, pixel_values, captions=None):
        return self.model(pixel_values=pixel_values).logits  # (B, C, H/4, W/4)


class MultimodalSegformer(nn.Module):
    """SegFormer encoder + CrossAttentionFusion (at last stage) + SegFormer decoder."""

    def __init__(self, segformer, bert, tokenizer, fusion):
        super().__init__()
        self.encoder     = segformer.segformer
        self.decode_head = segformer.decode_head
        self.bert        = bert           # kept frozen
        self.tokenizer   = tokenizer
        self.fusion      = fusion

    def _encode_text(self, captions, device):
        tokens = self.tokenizer(
            list(captions), return_tensors='pt',
            padding=True, truncation=True, max_length=128
        )
        tokens = {k: v.to(device) for k, v in tokens.items()}
        with torch.no_grad():
            return self.bert(**tokens).last_hidden_state  # (B, seq_len, 768)

    def forward(self, pixel_values, captions=None):
        device = pixel_values.device
        text_feat   = self._encode_text(captions, device)          # (B, L, 768)

        enc_out     = self.encoder(pixel_values, output_hidden_states=True)
        hidden      = list(enc_out.hidden_states)                  # 4 feature maps

        # Fuse at the last (most semantic) encoder stage
        last        = hidden[-1]                                   # (B, C, H/32, W/32)
        B, C, H, W  = last.shape
        flat        = last.flatten(2).transpose(1, 2)              # (B, H*W, C)
        fused, _    = self.fusion(flat, text_feat)                 # (B, H*W, C)
        hidden[-1]  = fused.transpose(1, 2).reshape(B, C, H, W)

        return self.decode_head(hidden)                            # (B, num_classes, H/4, W/4)


def build_model(config, device):
    """
    Factory function — builds the correct model for a given config.
    Returns a model with the unified interface:
        forward(pixel_values, captions=None) -> logits
    """
    segformer = SegformerForSemanticSegmentation.from_pretrained(
        config['backbone'],
        num_labels=config['num_classes'],
        id2label=id2label,
        label2id=label2id,
        ignore_mismatched_sizes=True,
    )

    if config['caption_col'] is None:
        model = SegformerWrapper(segformer)
    else:
        backbone_ch = BACKBONE_CHANNELS.get(config['backbone'], 256)
        tokenizer   = BertTokenizer.from_pretrained(config['text_model'])
        bert        = BertModel.from_pretrained(config['text_model'])
        if config['freeze_bert']:
            for p in bert.parameters():
                p.requires_grad = False
        fusion = CrossAttentionFusion(image_dim=backbone_ch, text_dim=768,
                                      hidden_dim=256, num_heads=4)
        model  = MultimodalSegformer(segformer, bert, tokenizer, fusion)

    return model.to(device)


## 3. Training Infrastructure

All training and evaluation logic lives in three functions ("train_one_epoch", "evaluate",
"run_experiment") so each experiment is a single function call with a different "CONFIG".
Mixed precision ("fp16") is enabled on CUDA only.

Evaluation metrics:

IoU (Intersection over Union) — per class
`IoU = intersection[Water] / union[Water]`

mIoU (Mean IoU): Average of IoU across all 7 classes:
`mIoU = (IoU_Tree + IoU_Shrub + IoU_Grass + IoU_Crop + IoU_Builtup + IoU_Barren + IoU_Water) / 7`

OA (Overall Accuracy)
`OA = correct pixels / total pixels`




In [ ]:
def evaluate(model, loader, device, config):
    """
    Computes mIoU, per-class IoU, and Overall Accuracy incrementally
    (intersection/union counts per batch) to avoid storing all predictions
    in RAM at once — 512px x 1000 samples would otherwise exhaust system RAM.
    Classes absent from both prediction and target are excluded from mIoU,
    matching standard segmentation benchmark practice.
    """
    model.eval()
    num_classes = config['num_classes']

    intersection = np.zeros(num_classes, dtype=np.int64)
    union        = np.zeros(num_classes, dtype=np.int64)
    correct      = 0
    total        = 0

    with torch.no_grad():
        for batch in loader:
            pv  = batch['pixel_values'].to(device)
            lbl = batch['labels']
            cap = batch.get('caption', None)

            logits    = model(pv, cap)
            upsampled = F.interpolate(logits, size=lbl.shape[-2:],
                                      mode='bilinear', align_corners=False)
            pred = upsampled.argmax(dim=1).cpu().numpy().flatten()
            tgt  = lbl.numpy().flatten()

            correct += (pred == tgt).sum()
            total   += len(tgt)

            for c in range(num_classes):
                pc = (pred == c)
                tc = (tgt  == c)
                intersection[c] += (pc & tc).sum()
                union[c]        += (pc | tc).sum()

    per_class_iou = [
        float(intersection[c] / union[c]) if union[c] > 0 else float('nan')
        for c in range(num_classes)
    ]
    valid = [v for v in per_class_iou if not np.isnan(v)]
    return {
        'miou':          float(np.mean(valid)) if valid else 0.0,
        'per_class_iou': per_class_iou,
        'oa':            float(correct / total),
    }


In [ ]:
def train_one_epoch(model, loader, optimizer, scaler, device, class_weights, config):
    """
    One training epoch with optional mixed precision (CUDA only), class-weighted loss,
    gradient accumulation, and tqdm progress bar.
    """
    from tqdm.auto import tqdm

    model.train()
    total_loss   = 0.0
    cw           = class_weights.to(device) if class_weights is not None else None
    accum_steps  = config.get('grad_accumulation', 1)

    pbar = tqdm(loader, desc='Training', leave=False)
    optimizer.zero_grad()

    for step, batch in enumerate(pbar):
        pv  = batch['pixel_values'].to(device)
        lbl = batch['labels'].to(device)
        cap = batch.get('caption', None)

        if scaler:  # CUDA fp16
            with torch.cuda.amp.autocast():
                logits = model(pv, cap)
                up     = F.interpolate(logits, size=lbl.shape[-2:],
                                       mode='bilinear', align_corners=False)
                loss   = F.cross_entropy(up, lbl, weight=cw) / accum_steps
            scaler.scale(loss).backward()
        else:
            logits = model(pv, cap)
            up     = F.interpolate(logits, size=lbl.shape[-2:],
                                   mode='bilinear', align_corners=False)
            loss   = F.cross_entropy(up, lbl, weight=cw) / accum_steps
            loss.backward()

        total_loss += loss.item() * accum_steps

        if (step + 1) % accum_steps == 0:
            if scaler:
                scaler.step(optimizer)
                scaler.update()
            else:
                optimizer.step()
            optimizer.zero_grad()

        pbar.set_postfix({'loss': f'{loss.item() * accum_steps:.4f}'})

    return total_loss / len(loader)


Logs 4 sample visualizations to W&B after training finishes so we can visually inspect what the model is predicting.

In [ ]:
def log_sample_predictions(model, dataset, device, n=4):
    """Log n sample predictions to W&B as a visual grid."""
    model.eval()
    images_to_log = []
    indices = np.linspace(0, len(dataset) - 1, n, dtype=int)

    for idx in indices:
        sample = dataset[idx]
        pv  = sample['pixel_values'].unsqueeze(0).to(device)
        lbl = sample['labels']
        cap = [sample['caption']] if 'caption' in sample else None

        with torch.no_grad():
            logits    = model(pv, cap)
            upsampled = F.interpolate(logits, size=lbl.shape,
                                      mode='bilinear', align_corners=False)
            pred = upsampled.argmax(dim=1).squeeze(0).cpu()

        mean = torch.tensor([0.485, 0.456, 0.406]).view(3,1,1)
        std  = torch.tensor([0.229, 0.224, 0.225]).view(3,1,1)
        img  = ((sample['pixel_values'].cpu() * std + mean) * 255).permute(1,2,0)
        img  = img.numpy().clip(0,255).astype(np.uint8)

        gt_rgb   = class_mask_to_rgb(lbl.numpy())
        pred_rgb = class_mask_to_rgb(pred.numpy())

        fig, axes = plt.subplots(1, 3, figsize=(12, 3), facecolor='white')
        for ax in axes:
            ax.set_facecolor('white')
            ax.axis('off')
        axes[0].imshow(img);      axes[0].set_title('Image',        color='black')
        axes[1].imshow(gt_rgb);   axes[1].set_title('Ground Truth', color='black')
        axes[2].imshow(pred_rgb); axes[2].set_title('Prediction',   color='black')
        plt.tight_layout()
        images_to_log.append(wandb.Image(fig, caption=sample['filename']))
        plt.close(fig)

    wandb.log({'sample_predictions': images_to_log})


def run_experiment(config, device, class_weights):
    """
    Build model, train for config['num_epochs'] epochs on the full training set,
    evaluate on val and test sets, log everything to W&B.
    Saves a complete results file to Drive using the BEST model weights (best val mIoU)
    so results can be loaded and analyzed in a separate Colab session without retraining.
    Returns a result dict stored in all_results for the comparison plots.
    """
    print(f'\n{"="*65}')
    print(f'  Run      : {config["run_name"]}')
    print(f'  Backbone : {config["backbone"]}')
    print(f'  Caption  : {config["caption_col"]}')
    print(f'  Epochs   : {config["num_epochs"]}   Image size: {config["image_size"]}')
    print(f'{"="*65}')

    if config['use_wandb']:
        wandb.init(
            project=config['wandb_project'],
            entity=config['wandb_entity'],
            name=config['run_name'],
            config=config,
            reinit=True
        )

    tl, vl, el = create_loaders(config, train_df, val_df, test_df, processor)

    model = build_model(config, device)
    n_trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f'  Trainable params: {n_trainable:,}')

    optimizer = optim.AdamW(
        [p for p in model.parameters() if p.requires_grad],
        lr=config['learning_rate'],
        weight_decay=config['weight_decay']
    )
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=config['num_epochs'])
    scaler = (
        torch.cuda.amp.GradScaler()
        if device.type == 'cuda' and config['mixed_precision'] else None
    )
    cw = class_weights if config['use_class_weights'] else None

    os.makedirs(config['save_dir'], exist_ok=True)
    best_val_miou  = 0.0
    best_val_metrics = None
    best_state_dict  = None
    history = {'train_loss': [], 'val_miou': [], 'val_oa': []}

    for epoch in range(1, config['num_epochs'] + 1):
        t_loss  = train_one_epoch(model, tl, optimizer, scaler, device, cw, config)
        metrics = evaluate(model, vl, device, config)
        scheduler.step()

        history['train_loss'].append(t_loss)
        history['val_miou'].append(metrics['miou'])
        history['val_oa'].append(metrics['oa'])

        if config['use_wandb']:
            log_dict = {
                'epoch': epoch, 'train/loss': t_loss,
                'val/miou': metrics['miou'], 'val/overall_accuracy': metrics['oa'],
                'lr': scheduler.get_last_lr()[0],
            }
            for i, name in enumerate(CLASS_NAMES):
                v = metrics['per_class_iou'][i]
                if not np.isnan(v):
                    log_dict[f'val/iou_{name}'] = v
            wandb.log(log_dict)

        print(f'  Epoch {epoch:02d}/{config["num_epochs"]}  '
              f'Loss: {t_loss:.4f}  Val mIoU: {metrics["miou"]:.4f}  OA: {metrics["oa"]:.4f}')

        if metrics['miou'] > best_val_miou:
            best_val_miou    = metrics['miou']
            best_val_metrics = metrics
            # Keep a copy of the best weights in memory
            best_state_dict  = {k: v.cpu().clone() for k, v in model.state_dict().items()
                                 if 'bert' not in k}
            print(f'           -> New best (mIoU={best_val_miou:.4f})')

    # Restore best weights into the model before test evaluation
    model.load_state_dict(best_state_dict, strict=False)
    print(f'\n  Restored best weights (val mIoU={best_val_miou:.4f}) for test evaluation.')

    # Test evaluation is done with the best model, not the final epoch
    test_metrics = evaluate(model, el, device, config)
    print(f'  Test mIoU: {test_metrics["miou"]:.4f}   OA: {test_metrics["oa"]:.4f}')

    # Save best weights + full history + test metrics to Drive
    results_path = os.path.join(config['save_dir'], f'{config["run_name"]}_results.pt')
    torch.save({
        'run_name':        config['run_name'],
        'caption_col':     config['caption_col'],
        'backbone':        config['backbone'],
        'config':          config,
        'history':         history,
        'best_val_miou':   best_val_miou,
        'best_val_metrics':best_val_metrics,
        'test_metrics':    test_metrics,
        # Best epoch weights — BERT excluded (frozen, reload from HuggingFace)
        # No need to save it, because we're using frozen weights
        'state_dict':      best_state_dict,
    }, results_path)
    print(f'  Results saved → {results_path}')

    if config['use_wandb']:
        wandb.log({
            'test/miou': test_metrics['miou'],
            'test/overall_accuracy': test_metrics['oa'],
            **{f'test/iou_{CLASS_NAMES[i]}': v
               for i, v in enumerate(test_metrics['per_class_iou'])
               if not np.isnan(v)}
        })
        log_sample_predictions(model, vl.dataset, device, n=4)
        wandb.finish()

    return {
        'run_name':      config['run_name'],
        'caption_col':   config['caption_col'],
        'backbone':      config['backbone'],
        'history':       history,
        'best_val_miou': best_val_miou,
        'test_metrics':  test_metrics,
        'model':         model,
    }


## 4. Experiments

Five experiments in total — three with MiT-B0 to isolate the text fusion effect, then
two backbone scaling experiments with the best-performing caption source (Gemma).

| Exp | Backbone | Caption | Purpose |
|---|---|---|---|
| 1 | MiT-B0 | None | Image-only baseline |
| 2 | MiT-B0 | `vision_gemma3-4b` | Does text fusion help? |
| 3 | MiT-B0 | `vision_qwen3-vl-8b` | Gemma vs Qwen captions |
| 4 | MiT-B1 | `vision_gemma3-4b` | Does larger backbone help? |
| 5 | MiT-B2 | `vision_gemma3-4b` | How far does scaling go? |


In [ ]:
# Following was required because firstly Colab worked really really slow
# Because it tries to reach my folders in each batch --> it takes too much time
# So we need to copy to the local disk
import shutil
print('Copying data from Drive to local disk...')
shutil.copytree(
    '/content/drive/MyDrive/multimodal-landcover-segmentation/data',
    '/content/data'
)
print('Done! Updating paths...')

DATA_DIR   = '/content/data'
IMAGES_DIR = '/content/data/images'
MASKS_DIR  = '/content/data/masks'
CAPTIONS_CSV = '/content/data/captions.csv'
print('Data is now on local SSD.')

In [ ]:
# Experiment 1: Image-only Baseline
config_baseline = {
    **CONFIG,
    'run_name':             'baseline-mit-b0',
    'caption_col':          None,
    'image_size':           512,
    'batch_size':           4,
    'grad_accumulation':    2,
    'num_epochs':           20,
    'mixed_precision':      True,
}

all_results = {}  # accumulates results across all experiments
all_results['baseline'] = run_experiment(config_baseline, device, class_weights)


In [ ]:
# Experiment 2: Fused Model — vision_gemma3-4b captions
config_gemma = {
    **CONFIG,
    'run_name':       'fused-gemma',
    'caption_col':    'vision_gemma3-4b',
    'image_size':     512,
    'batch_size':     4,
    'num_epochs':     20,
    'mixed_precision': True,
}

all_results['fused_gemma'] = run_experiment(config_gemma, device, class_weights)


In [ ]:
# Experiment 3: Fused Model — vision_qwen3-vl-8b captions
config_qwen = {
    **CONFIG,
    'run_name':       'fused-qwen',
    'caption_col':    'vision_qwen3-vl-8b',
    'image_size':     512,
    'batch_size':     4,
    'num_epochs':     20,
    'mixed_precision': True,
}

all_results['fused_qwen'] = run_experiment(config_qwen, device, class_weights)


In [ ]:
# Experiment 4: Fused Model — MiT-B1 + vision_gemma3-4b
# MiT-B1 (apprx 14M params, 512 last-stage channels) —>>> first backbone scaling step.
# Main aim is to see: does a larger encoder improve land cover segmentation with text fusion?
config_b1_gemma = {
    **CONFIG,
    'run_name':          'fused-b1-gemma',
    'caption_col':       'vision_gemma3-4b',
    'backbone':          'nvidia/mit-b1',
    'image_size':        512,
    'batch_size':        4,
    'grad_accumulation': 2,
    'num_epochs':        20,
    'mixed_precision':   True,
}

all_results['fused_b1_gemma'] = run_experiment(config_b1_gemma, device, class_weights)


In [ ]:
# Experiment 5: Fused Model — MiT-B2 + vision_gemma3-4b
# MiT-B2 (apprx 27M params, 512 last-stage channels) —>> strong accuracy/speed tradeoff,
# commonly used as the research-grade backbone in segmentation benchmarks.
# Main aim is to see: how far does backbone scaling go with text fusion?
config_b2_gemma = {
    **CONFIG,
    'run_name':          'fused-b2-gemma',
    'caption_col':       'vision_gemma3-4b',
    'backbone':          'nvidia/mit-b2',
    'image_size':        512,
    'batch_size':        4,
    'grad_accumulation': 2,
    'num_epochs':        20,
    'mixed_precision':   True,
}

all_results['fused_b2_gemma'] = run_experiment(config_b2_gemma, device, class_weights)


## 5. Results and Analysis


In [ ]:
# Load saved results (run this in a NEW session instead of re-training ofc)
# Run this cell to reconstruct all_results from the saved .pt files on Drive.
# Skip this cell if  finished training for all experiments in the same session.
# But unfortunately it is no possible by using Colab :/
# So we need to run the following

CKPT_DIR = '/content/drive/MyDrive/multimodal-landcover-segmentation/checkpoints'

RUN_CONFIGS = {
    'baseline':      {'run_name': 'baseline-mit-b0', 'caption_col': None},
    'fused_gemma':   {'run_name': 'fused-gemma',      'caption_col': 'vision_gemma3-4b'},
    'fused_qwen':    {'run_name': 'fused-qwen',       'caption_col': 'vision_qwen3-vl-8b'},
    'fused_b1_gemma':{'run_name': 'fused-b1-gemma',   'caption_col': 'vision_gemma3-4b'},
    'fused_b2_gemma':{'run_name': 'fused-b2-gemma',   'caption_col': 'vision_gemma3-4b'},
}

all_results = {}

for key, info in RUN_CONFIGS.items():
    results_path = os.path.join(CKPT_DIR, f'{info["run_name"]}_results.pt')
    if not os.path.exists(results_path):
        print(f'  SKIPPING {key} — file not found: {results_path}')
        continue

    saved = torch.load(results_path, map_location=device)
    config = saved['config']

    # Rebuild model and load best weights
    model = build_model(config, device)
    missing, unexpected = model.load_state_dict(saved['state_dict'], strict=False)
    print(f'  Loaded {key}: missing={len(missing)} unexpected={len(unexpected)}')

    all_results[key] = {
        'run_name':      saved['run_name'],
        'caption_col':   saved['caption_col'],
        'backbone':      saved['backbone'],
        'history':       saved['history'],
        'best_val_miou': saved['best_val_miou'],
        'test_metrics':  saved['test_metrics'],
        'model':         model,
    }
    print(f'  {key}: best_val_mIoU={saved["best_val_miou"]:.4f}  '
          f'test_mIoU={saved["test_metrics"]["miou"]:.4f}')

print(f'\nLoaded {len(all_results)} experiments: {list(all_results.keys())}')


In [ ]:
# Summary results table
print(f'\n{"Model":<30} {"Best Val mIoU":>15} {"Test mIoU":>12} {"Test OA":>10}')
print('-' * 70)
for key, res in all_results.items():
    tm = res['test_metrics']
    print(f'{res["run_name"]:<30} {res["best_val_miou"]:>15.4f} '
          f'{tm["miou"]:>12.4f} {tm["oa"]:>10.4f}')

baseline_miou = all_results['baseline']['test_metrics']['miou']
for key in ['fused_gemma', 'fused_qwen']:
    if key in all_results:
        delta = all_results[key]['test_metrics']['miou'] - baseline_miou
        print(f'  Δ mIoU vs baseline ({key}): {delta:+.4f}')


In [ ]:
STYLES = [
    ('o-',  'black',    'B0 — Image-only (baseline)'),
    ('s--', 'black',    'B0 + Gemma'),
    ('^:',  'dimgray',  'B0 + Qwen'),
    ('D-.', 'black',    'B1 + Gemma'),
    ('P-',  'dimgray',  'B2 + Gemma'),
]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5), facecolor='white')

for ax in (ax1, ax2):
    ax.set_facecolor('white')
    for spine in ax.spines.values():
        spine.set_edgecolor('black')
    ax.tick_params(colors='black')
    ax.xaxis.label.set_color('black')
    ax.yaxis.label.set_color('black')
    ax.title.set_color('black')

for (style, color, label), res in zip(STYLES, all_results.values()):
    epochs = list(range(1, len(res['history']['train_loss']) + 1))
    ax1.plot(epochs, res['history']['train_loss'], style, color=color,
             lw=2, ms=5, label=label)
    ax2.plot(epochs, res['history']['val_miou'], style, color=color,
             lw=2, ms=5, label=label)

ax1.set_xlabel('Epoch'); ax1.set_ylabel('Cross-Entropy Loss')
ax1.set_title('Training Loss'); ax1.legend(labelcolor='black', fontsize=8)
ax1.grid(True, linestyle='--', alpha=0.4, color='gray')

ax2.set_xlabel('Epoch'); ax2.set_ylabel('Mean IoU')
ax2.set_title('Validation mIoU'); ax2.legend(labelcolor='black', fontsize=8)
ax2.grid(True, linestyle='--', alpha=0.4, color='gray')

fig.suptitle('Phase 2 — All Experiments: Baseline vs Multimodal SegFormer',
             fontsize=11, color='black')
plt.tight_layout()
os.makedirs('results/figures', exist_ok=True)
plt.savefig('results/figures/phase2_training_curves.png', dpi=300,
            bbox_inches='tight', facecolor='white')
plt.show()


In [ ]:
HATCHES = ['', '///', 'xxx', '...', '+++']

n_exps = len(all_results)
width  = 0.8 / n_exps

fig, ax = plt.subplots(figsize=(13, 5), facecolor='white')
ax.set_facecolor('white')
for spine in ax.spines.values():
    spine.set_edgecolor('black')
ax.tick_params(colors='black')

x = np.arange(CONFIG['num_classes'])

for i, (res, hatch) in enumerate(zip(all_results.values(), HATCHES)):
    ious = [v if not np.isnan(v) else 0.0
            for v in res['test_metrics']['per_class_iou']]
    ax.bar(x + i * width, ious, width=width, label=res['run_name'],
           color='white', edgecolor='black', hatch=hatch)

ax.set_xticks(x + width * (n_exps - 1) / 2)
ax.set_xticklabels(CLASS_NAMES, rotation=20, ha='right', color='black')
ax.set_ylabel('IoU', color='black')
ax.set_title('Per-Class IoU on Test Set — All Experiments', color='black')
ax.legend(labelcolor='black', fontsize=8)
ax.grid(True, axis='y', linestyle='--', alpha=0.4, color='gray')
ax.set_ylim(0, 1)

plt.tight_layout()
plt.savefig('results/figures/phase2_per_class_iou.png', dpi=300,
            bbox_inches='tight', facecolor='white')
plt.show()


In [ ]:
# Sample predictions — compare all three models on the same validation images.
# I pick 3 samples that show different dominant land cover types.
SAMPLE_INDICES = [0, len(val_df) // 2, len(val_df) - 1]

_, vl_base, _ = create_loaders(config_baseline, train_df, val_df, test_df, processor)

for sample_idx in SAMPLE_INDICES:
    sample = vl_base.dataset[sample_idx]
    pv  = sample['pixel_values'].unsqueeze(0).to(device)
    lbl = sample['labels']

    fig, axes = plt.subplots(1, len(all_results) + 2,
                             figsize=(4 * (len(all_results) + 2), 3.5),
                             facecolor='white')

    # Input image
    mean = torch.tensor([0.485, 0.456, 0.406]).view(3,1,1)
    std  = torch.tensor([0.229, 0.224, 0.225]).view(3,1,1)
    img_np = ((sample['pixel_values'].cpu() * std + mean) * 255)\
             .permute(1,2,0).numpy().clip(0,255).astype(np.uint8)
    axes[0].imshow(img_np); axes[0].set_title('Input Image', color='black')

    # Ground truth
    axes[1].imshow(class_mask_to_rgb(lbl.numpy()))
    axes[1].set_title('Ground Truth', color='black')

    # Each model's prediction
    for ax, (key, res) in zip(axes[2:], all_results.items()):
        model = res['model']
        model.eval()
        # Build appropriate caption input
        cfg = config_baseline if res['caption_col'] is None else \
              (config_gemma if res['caption_col'] == 'vision_gemma3-4b' else config_qwen)
        _, vl_exp, _ = create_loaders(cfg, train_df, val_df, test_df, processor)
        s_exp = vl_exp.dataset[sample_idx]
        cap = [s_exp['caption']] if 'caption' in s_exp else None

        with torch.no_grad():
            logits = model(pv, cap)
            up     = F.interpolate(logits, size=lbl.shape,
                                   mode='bilinear', align_corners=False)
            pred   = up.argmax(dim=1).squeeze(0).cpu()

        ax.imshow(class_mask_to_rgb(pred.numpy()))
        ax.set_title(res['run_name'], color='black', fontsize=8)

    for ax in axes:
        ax.axis('off')
        ax.set_facecolor('white')

    patches = [mpatches.Patch(color=np.array(CLASS_INFO[i]['color'])/255,
                               label=CLASS_INFO[i]['name'])
               for i in range(CONFIG['num_classes'])]
    fig.legend(handles=patches, loc='lower center', ncol=7, fontsize=7,
               bbox_to_anchor=(0.5, -0.04), labelcolor='black')
    plt.tight_layout()
    plt.savefig(f'results/figures/phase2_pred_sample{sample_idx}.png',
                dpi=300, bbox_inches='tight', facecolor='white')
    plt.show()


## Summary — Phase 2

| What was done         | Detail                                                             |
|-----------------------|--------------------------------------------------------------------|
| Full dataset          | 8,000 train / 1,000 val / 1,000 test (80/10/10, seed=42)           |
| Class-weighted loss   | Inverse-frequency weights address dominant-class bias from Phase 1 |
| Five experiments      | Baseline, Fused-Gemma, Fused-Qwen, Fused B1+Gemma, Fused B2-Gemma  |
| W&B logging           | Loss, mIoU, OA, per-class IoU, LR, sample predictions              |

### Phase 3 Possible Ablation Plan (But it can be changed, not certain)

- **Random caption:** Replace correct caption with a random mismatched one — does mIoU drop?
  If it does, the model is genuinely using text information.
- **Caption source:** Gemma vs Qwen — which VLM produces more useful descriptions?
- **Frozen vs fine-tuned BERT:** Does unfreezing BERT improve results?
- **Fusion stage:** Last encoder stage vs middle stage — where is fusion most effective?
